# Phase 7 · Prior Case Retrieval — Architecture-1 decoder + LoRMA retriever, fused with the learned re-ranker

| | |
|---|---|
| **Codabench** | Micro-F1@7 **0.466** · Micro-P@7 0.4756 · Micro-R@7 0.4569 (rank 6 of 6 in Phase 7) |
| **Own dev / validation** | dev micro-F1@7: fused 0.4769 (model weight 0.4) · model alone 0.3702 · re-ranker 0.4725 · BM25 0.4647 |
| **Backbone** | Architecture-1 decoder-only (110M) |

**The task, in simple words:** for a query judgment, find the **7 earlier cases it most likely cites**, from that split's candidate pool.

**How this notebook solves it:**
1. Every sentence with a `<CITATION>` (± one sentence) becomes a small search query; candidates are read as 200-word passages.
2. Date filter: a judgment can only cite older cases, so newer candidates are never ranked.
3. BM25 word overlap + a small learned re-ranker give a strong word-based ranking.
4. The Phase-1 model is trained to match citation contexts with passages of the cited case.
5. Final ranking = model score mixed with the re-ranker score (mix chosen on dev); the top 7 are submitted.

**What is special in this run:** **LoRMA** — the Phase-1 weights stay frozen. Every layer gets a small learned *multiplier* for its output, `(I + B·A)·W`, so only about 3% new weights are trained. B starts at 0, so training starts from exactly the Phase-1 model.

*Exact setting:* LoRMA, rank 16, α 32, on qkv / out_proj / gate / up / down of all 12 blocks (3.05M weights, 2.71%) + retriever head (0.59M).

**To run it:** Attach the tokenizer notebook's output and the Phase-1 checkpoint as Kaggle inputs, add the Hugging Face token as the Kaggle secret **`HF`**, pick a GPU (T4), then *Run All*. Paths are in the configuration cell; if a checkpoint is not found there, the notebook searches `/kaggle/input`. Test labels are never used.

Every step below starts with a short explanation: first in simple words, then how it works exactly.

### CELL 1 · Imports + reproducibility + device

**In simple words:** Load the Python libraries, fix the random seed (so a re-run gives the same result) and pick the GPU.

**How it works:** Seeds for `random`, NumPy and PyTorch; `DEVICE` is the GPU when one is attached.

In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import re
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np
from scipy import sparse

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


### CELL 2 · Configuration

**In simple words:** Every setting of this run in one place: input paths, sizes, learning rates, epochs. To change the experiment, change only this cell.

**How it works:** The comment block at the top summarises the whole method. Mixed precision: float16 + GradScaler on T4 / P100, bfloat16 on A100. `DEBUG_MODE = True` (where present) runs a few-minute check on a small subset.

In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# Phase 7 (Prior Case Retrieval): Architecture-1 decoder + LoRMA
#
# What the data shows (dev micro-F1@7 of plain word overlap, BM25):
#   whole query vs whole candidate 0.157 | + date filter 0.203 | + every <CITATION> sentence of the
#   query as its own small query 0.385 | + every candidate read as passages (best passage counts) 0.466
# So this notebook works the same way, with the Phase-1 model:
#   query     -> its CITATION CONTEXTS (the sentence with a <CITATION> + one sentence before / after),
#                one chunk each (no <CITATION> at all: passages of the whole query)
#   candidate -> PASSAGES of PASSAGE_WORDS words (up to MAX_CHUNKS, spread over the case), one chunk each
#   chunk     -> Phase-1 model -> token vectors (learned mix of ALL layers), standardized
#                -> mean over the tokens -> projection (starts as the identity) -> unit vector
#   score(context, candidate) = best cosine over the candidate's passages; standardized over the pool
#   score(query, candidate)   = best score over the query's contexts
#   date filter: a case only cites OLDER cases -> candidates decided after the query are never ranked
#   training : contrastive (InfoNCE) — the relevant candidate against BM25 hard negatives and the other
#              candidates of the batch; TRAIN queries only; the best epoch = the best dev F1@7 of the MODEL
#   re-ranker: logistic regression on 9 BM25-based signals of the BM25 top 100 (best / 2nd-best context,
#              whole case, years between the cases, Supreme Court judgment, length, ...), fitted on TRAIN
#              queries only (CELL 10): +0.014 F1@7 over BM25 on held-out train queries, +0.008 on dev
#   ranking  : the model's score fused with the re-ranker's score (weight picked on dev)
#
# Version 3 changes (after the first real runs: model alone 0.36, fused 0.466 = BM25 0.465):
#   - the best epoch was picked by the FUSED dev F1, which hardly moves (BM25 dominates it), so training
#     kept epoch 2 and stopped early while the model was still improving -> now picked by the model's own
#     dev F1, more epochs, PATIENCE 3
#   - fusing ANY dense model with BM25 adds little here (checked on the real data with a stronger
#     bag-of-words retriever: 0.40 alone, +0.004 fused), so the gain comes from the learned re-ranker
#   - REUSE_TRAINED = True loads the retriever.pt of an earlier run of this notebook instead of training
#
#   model    : Architecture-1 decoder + LoRMA
# ============================================================

RUN_NAME = "p7_arch1_lorma"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)
CHECKPOINT_PATH = Path(
    "/kaggle/input/notebooks/iitianritik/decoder-only-archi-1/legal_lm/best_model.pt"
)
LOCAL_DATA_DIR = None        # a folder with Train/ Dev/ Test/ (None = download from Hugging Face)
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

OUTPUT_DIR = Path(f"/kaggle/working/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# How queries and candidates are read
# ------------------------------------------------------------

CITATION_WINDOW = 1          # sentences kept before / after a <CITATION> sentence
MAX_CONTEXTS = 48            # citation contexts per query (more: evenly spread)
PASSAGE_WORDS = 200          # words per candidate passage (~ 480 tokens with the Phase-1 tokenizer)
MAX_CHUNKS = 16              # passages per candidate for ranking (more: evenly spread over the case)
CHUNK_LEN = 512              # tokens per chunk (a longer passage / context is cut)
DATE_FILTER = True           # never rank a candidate decided after the query
POOL_SKIP_SPACES = True      # whitespace tokens stay in the input but are not pooled
POOL_SKIP_TOKENS = 4         # the first tokens of a chunk are not pooled (they behave abnormally in LMs)

# ------------------------------------------------------------
# LoRMA (Bihany, Patel & Modi, 2025):  W' = (I + alpha/r * B A) W,  B = 0 at the start
# Phase-1 weights stay FROZEN; only A, B (+ the head) train
# ------------------------------------------------------------

LORMA_TARGETS = ["attn.qkv", "attn.out_proj", "mlp.gate", "mlp.up", "mlp.down"]   # every linear layer of every block
LORMA_RANK = 16
LORMA_ALPHA = 32
LORMA_DROPOUT = 0.0          # 0: both GradCache passes must compute the same vectors
LR_PEFT = 2e-4


# ------------------------------------------------------------
# Retrieval head + training (TRAIN queries only)
# ------------------------------------------------------------

TEMPERATURE = 0.05           # cosine / TEMPERATURE in the contrastive loss
LR_HEAD = 5e-4               # layer mix + projection

REUSE_TRAINED = False        # True: load retriever.pt of an earlier run of THIS notebook (attach that
                             # notebook's output as an input) instead of training
EPOCHS = 6
BATCH_QUERIES = 8            # queries per weight update
HARD_NEGATIVES = 3           # BM25 hard negatives per query (+ the other candidates of the batch)
HARD_POOL = 30               # hard negatives are drawn from the BM25 top 30 (relevant / filtered cases removed)
TRAIN_CONTEXTS = 8           # citation contexts per query in a training step (random subset)
TRAIN_CHUNKS = 4             # passages per candidate in a training step: its best BM25 passages for that query
EPOCH_EVAL_CHUNKS = 8        # passages per candidate for the dev check after every epoch (faster than MAX_CHUNKS)
EMBED_GROUP_CHUNKS = 16      # chunks per forward pass WITH gradients (more = a little faster, more memory)
EVAL_CHUNKS = 64             # chunks per forward pass without gradients
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0
PATIENCE = 3                 # stop if the model's dev F1@7 does not improve for 3 epochs
STATS_DOCS = 300             # train candidates for the token statistics (standardization) before training


# ------------------------------------------------------------
# Ranking
# ------------------------------------------------------------

TOP_K = 7                                                       # the leaderboard reads the first 7 candidates
FUSION_WEIGHTS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]   # weight of the MODEL score (rest = re-ranker), picked on dev
RERANK_TOP = 100             # the re-ranker scores the BM25 top 100 of every query (the rest is never predicted)
RERANK_C = 1.0               # inverse regularization strength of the logistic regression (0.01 - 10 all give the same F1)
BM25_K1 = 1.2
BM25_B = 0.75


# ------------------------------------------------------------
# Debug mode (quick run on a few queries)
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_QUERIES = 40           # per split
DEBUG_EXTRA_CANDIDATES = 300 # random candidates kept in every pool (+ the relevant ones)

if DEBUG_MODE:
    EPOCHS = 2
    STATS_DOCS = 50


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


assert 0 < min(FUSION_WEIGHTS) and TRAIN_CHUNKS <= MAX_CHUNKS

print("Run       :", RUN_NAME)
print("Query     :", f"<CITATION> sentences +-{CITATION_WINDOW} (up to {MAX_CONTEXTS}), one chunk each")
print("Candidate :", f"passages of {PASSAGE_WORDS} words (up to {MAX_CHUNKS}), one chunk each")
print("Batch     :", f"{BATCH_QUERIES} queries x (1 relevant + {HARD_NEGATIVES} hard negatives) per update")
print("Output    :", OUTPUT_DIR)
print("AMP dtype :", AMP_DTYPE if USE_AMP else "disabled")

Run       : p7_arch1_lorma
Query     : <CITATION> sentences +-1 (up to 48), one chunk each
Candidate : passages of 200 words (up to 16), one chunk each
Batch     : 8 queries x (1 relevant + 3 hard negatives) per update
Output    : /kaggle/working/p7_arch1_lorma
AMP dtype : torch.float16


### CELL 3 · Load the Phase-1 tokenizer + fast encoding

**In simple words:** Load our own Phase-1 BPE tokenizer — the same one the Phase-1 model was trained with. Any other tokenizer would give the model token ids it has never seen.

**How it works:** `tokenizer.py` from the tokenizer notebook. Long documents: only the needed part is tokenized, and every pre-token piece is BPE-encoded once and cached, which makes tokenizing thousands of judgments fast.

In [3]:
# ============================================================
# CELL 3 — LOAD THE PHASE-1 TOKENIZER + FAST ENCODING
#
# Cases are long, so only the parts we need are tokenized,
# and every pre-token piece is BPE-encoded only once (cache).
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


piece_cache = {}      # pre-token piece -> token ids


def encode_fast(text):
    """Same ids as tokenizer.encode(text), but every piece is encoded only once."""

    ids = []

    for piece in tok_module.PRETOKEN_PATTERN.findall(tok_module.normalize_text(text)):

        if piece not in piece_cache:
            piece_cache[piece] = tokenizer.encode(piece)

        ids.extend(piece_cache[piece])

    return ids


def head_tokens(text, n):
    """First n tokens of text (tokenizes only the beginning, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[:n]

        part = text[:chars]
        part = part[:max(part.rfind(" "), 1)]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[:n]

        chars *= 2


demo = "The appeal is allowed. The order of the High Court dated 12.03.2010 is set aside."
assert encode_fast(demo) == tokenizer.encode(demo)
print("Demo:", [tokenizer.id_to_token[i] for i in encode_fast(demo)][:12])


# masked parts of the cases look like <CITATION>: they become one ordinary word ("citation")
TAG_PATTERN = re.compile(r"<([A-Za-z_]+)>")


def clean(text):
    return TAG_PATTERN.sub(lambda m: m.group(1).lower().replace("_", " "), text)


# whitespace-only tokens (" ", "\n", ...): kept in the input, not pooled (POOL_SKIP_SPACES)
def token_text(i):
    try:
        return tokenizer.id_to_token[i]
    except (KeyError, IndexError):
        return None


SPACE_TOKEN_IDS = [i for i in range(VOCAB_SIZE)
                   if i != PAD_ID and isinstance(token_text(i), str) and token_text(i).strip() == ""]

print(f"Whitespace-only tokens in the vocabulary: {len(SPACE_TOKEN_IDS)}")

Vocabulary size: 32004 | PAD / UNK: 0 1
Demo: ['The', ' ', 'appeal', ' ', 'is', ' ', 'allow', 'ed', '.', ' ', 'The', ' ']
Whitespace-only tokens in the vocabulary: 3


### CELL 4 · Phase-7 data files (download needs the Kaggle secret "HF")

**In simple words:** Download the official data of this phase from Hugging Face.

**How it works:** The dataset is gated: the token comes from the Kaggle secret `HF` (never written in the notebook). Files are cached, so a re-run does not download again.

In [4]:
# ============================================================
# CELL 4 — PHASE-7 DATA FILES  (download needs the Kaggle secret "HF")
# ============================================================

PHASE7_FILES = {
    "train":            "Phase_7_PCR/Train/train.json",
    "train_candidates": "Phase_7_PCR/Train/train_candidates.json",
    "dev":              "Phase_7_PCR/Dev/dev.json",
    "dev_candidates":   "Phase_7_PCR/Dev/dev_candidates.json",
    "test":             "Phase_7_PCR/Test/test.json",
    "test_candidates":  "Phase_7_PCR/Test/test_candidates.json",
}

DATA_FILES = {}

if LOCAL_DATA_DIR is not None:

    for name, filename in PHASE7_FILES.items():
        DATA_FILES[name] = Path(LOCAL_DATA_DIR) / filename.replace("Phase_7_PCR/", "")

else:

    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    for name, filename in PHASE7_FILES.items():
        DATA_FILES[name] = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                                           repo_type="dataset", token=HF_TOKEN)

for name, path in DATA_FILES.items():
    print(f"{name:17s} -> {path}  ({os.path.getsize(path) / 1e6:.0f} MB)")

train             -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Train/train.json  (33 MB)
train_candidates  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Train/train_candidates.json  (160 MB)
dev               -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Dev/dev.json  (6 MB)
dev_candidates    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Dev/dev_candidates.json  (52 MB)
test              -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_7_PCR/Test/test.json  (9 MB)
test_candidates   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS

### CELL 5 · Load queries + candidate pools + years (date filter) + statistics

**In simple words:** Read the queries and every split's candidate pool, and find the year of every case. A judgment can only cite **older** cases, so newer candidates are never ranked (the date filter).

**How it works:** Year = the `DATE OF JUDGMENT` header, otherwise the latest year mentioned (the `CITATOR INFO` part, which lists later cases, is skipped). The filter removes ≈ 43% of every pool and loses ≈ 7% of the relevant pairs.

In [5]:
# ============================================================
# CELL 5 — LOAD QUERIES + CANDIDATE POOLS + YEARS (date filter) + STATISTICS
#
# Every split has its own pool: train queries are ranked against train_candidates,
# dev against dev_candidates, test against test_candidates.
# Each case is {id: {"text": [sentences], "relevant_candidates": [...]}} (the loader also
# accepts a list of records and text as one string).
# ============================================================

ID_KEYS = ["id", "query_id", "candidate_id", "case_id", "doc_id", "_id"]
TEXT_KEYS = ["text", "sentences", "document", "content", "judgment", "body"]
REL_KEYS = ["relevant_candidates", "relevant", "relevant_ids", "positives", "labels", "citations"]
SENTENCE_END = re.compile(r"(?<=[.?!])\s+(?=[A-Z(<0-9])")


def first_key(record, keys):
    for k in keys:
        if k in record:
            return k
    return None


def as_sentences(value):
    """List of sentences: a list is kept as it is (the data's own sentences); one string is split at sentence ends."""
    if isinstance(value, list):
        return [v.strip() if isinstance(v, str) else " ".join(as_sentences(v)) for v in value if v]
    if isinstance(value, dict):
        key = first_key(value, TEXT_KEYS)
        return as_sentences(value[key]) if key else []
    if value is None:
        return []
    return [s for s in SENTENCE_END.split(str(value)) if s.strip()]


def canon(case_id):
    """'0000590455', '590455' and 590455 are the same case."""
    return str(case_id).strip().lstrip("0") or "0"


def load_cases(path):
    """-> list of {"id", "sentences", "relevant"} (relevant = list of ids, or None if the file has none)."""

    data = json.load(open(path, encoding="utf-8"))

    if isinstance(data, dict) and len(data) == 1 and isinstance(next(iter(data.values())), list):
        data = next(iter(data.values()))                 # {"data": [...]} wrapper

    if isinstance(data, dict):                           # {id: record} or {id: text}
        records = []
        for case_id, value in data.items():
            record = dict(value) if isinstance(value, dict) else {"text": value}
            record.setdefault("id", case_id)
            records.append(record)
    else:
        records = data

    cases = []
    for record in records:

        id_key, text_key, rel_key = first_key(record, ID_KEYS), first_key(record, TEXT_KEYS), first_key(record, REL_KEYS)
        assert id_key and text_key, f"{path}: unknown record format, keys = {list(record)[:12]}"

        relevant = record.get(rel_key) if rel_key else None
        if isinstance(relevant, (str, int)):
            relevant = [relevant]

        cases.append({"id": str(record[id_key]).strip(), "sentences": as_sentences(record[text_key]),
                      "relevant": None if relevant is None else [str(r).strip() for r in relevant]})

    return cases


# ------------------------------------------------------------
# Year of a case: "DATE OF JUDGMENT: dd/mm/yyyy" if the header has it, otherwise the latest
# year the text mentions (a judgment cannot mention a later year). The "CITATOR INFO" part of a
# header lists LATER cases that cite this one, so it is left out.
# ------------------------------------------------------------

DATE_PATTERN = re.compile(r"DATE OF JUDGMENT\s*:?\s*\d{1,2}[/.-]\d{1,2}[/.-](\d{4})")
YEAR_PATTERN = re.compile(r"\b(1[89]\d\d|20[0-4]\d)\b")
SUPREME_COURT_HEADER = re.compile(r"PETITIONER:|DATE OF JUDGMENT")    # header of the Supreme Court judgments


def case_year(sentences):
    text = " ".join(sentences)
    m = DATE_PATTERN.search(text[:3000])
    if m:
        return int(m.group(1))
    i = text.find("CITATOR INFO")
    if i >= 0:
        j = text.find("ACT:", i)
        text = text[:i] + text[j if j > 0 else i + 600:]
    years = [int(y) for y in YEAR_PATTERN.findall(text)]
    return max(years) if years else None


SPLITS = {}

for split in ["train", "dev", "test"]:

    queries = load_cases(DATA_FILES[split])
    pool = load_cases(DATA_FILES[f"{split}_candidates"])

    seen, unique_pool = set(), []                        # one entry per candidate id
    for c in pool:
        if canon(c["id"]) not in seen:
            seen.add(canon(c["id"]))
            unique_pool.append(c)
    pool = unique_pool

    index = {canon(c["id"]): k for k, c in enumerate(pool)}
    missing = 0

    for d in queries + pool:
        d["year"] = case_year(d["sentences"])
        d["words"] = sum(len(s.split()) for s in d["sentences"])                          # (re-ranker, CELL 10)
        d["supreme_court"] = bool(SUPREME_COURT_HEADER.search(" ".join(d["sentences"])[:1500]))

    pool_years = np.array([c["year"] if c["year"] is not None else -1 for c in pool])

    for q in queries:
        q["self"] = index.get(canon(q["id"]))            # the query itself, if it is in its own pool

        banned = set() if q["self"] is None else {q["self"]}
        if DATE_FILTER and q["year"] is not None:
            banned |= set(np.where(pool_years > q["year"])[0].tolist())
        q["banned"] = np.array(sorted(banned), dtype=np.int64)       # never ranked for this query

        if split == "test":
            q["relevant"] = None                         # test labels are private: never read
        if q["relevant"] is None:
            q["gold"] = None
            continue
        gold = [index[canon(r)] for r in q["relevant"] if canon(r) in index]
        missing += len(q["relevant"]) - len(gold)
        q["all_gold"] = np.array(sorted(set(gold) - {q["self"]}), dtype=np.int64)
        q["gold"] = np.setdiff1d(q["all_gold"], q["banned"])         # relevant cases the ranking can still reach

    SPLITS[split] = {"queries": queries, "pool": pool, "missing": missing}


if DEBUG_MODE:
    rng = random.Random(SEED)
    for split, S in SPLITS.items():
        queries = S["queries"][:DEBUG_QUERIES]
        keep = set(rng.sample(range(len(S["pool"])), min(DEBUG_EXTRA_CANDIDATES, len(S["pool"]))))
        for q in queries:
            keep |= set(q["all_gold"].tolist()) if q["gold"] is not None else set()
            keep |= {q["self"]} if q["self"] is not None else set()
        keep = sorted(keep)
        new_index = {old: new for new, old in enumerate(keep)}
        for q in queries:
            q["banned"] = np.array([new_index[b] for b in q["banned"] if b in new_index], dtype=np.int64)
            for key in ["gold", "all_gold"]:
                if q.get(key) is not None:
                    q[key] = np.array([new_index[g] for g in q[key]], dtype=np.int64)
            q["self"] = new_index.get(q["self"])
        SPLITS[split] = {"queries": queries, "pool": [S["pool"][k] for k in keep], "missing": S["missing"]}


def oracle_f1(queries, key="all_gold"):
    n_rel = np.array([len(q[key]) for q in queries])
    hits = np.minimum(n_rel, TOP_K).sum()
    p, r = hits / (TOP_K * len(queries)), hits / max(n_rel.sum(), 1)
    return 2 * p * r / (p + r) if p + r else 0.0


print(f"{'split':6s} {'queries':>8s} {'pool':>6s} {'relevant/query':>15s} {'words/case':>16s} "
      f"{'with a year':>12s} {'oracle F1@7':>12s}")

for split, S in SPLITS.items():
    queries, pool = S["queries"], S["pool"]
    words = lambda d: sum(len(s.split()) for s in d["sentences"])
    with_year = np.mean([d["year"] is not None for d in queries + pool])
    if queries[0]["gold"] is not None:
        n_rel = np.array([len(q["all_gold"]) for q in queries])
        rel_text, oracle_text = f"{n_rel.mean():.2f} (max {n_rel.max()})", f"{oracle_f1(queries):.4f}"
    else:
        rel_text, oracle_text = "(private)", "-"
    print(f"{split:6s} {len(queries):8d} {len(pool):6d} {rel_text:>15s} "
          f"{np.median([words(q) for q in queries]):7.0f} / {np.median([words(c) for c in pool]):6.0f} "
          f"{with_year:12.1%} {oracle_text:>12s}")
    if S["missing"]:
        print(f"   ⚠️ {split}: {S['missing']} relevant ids are not in the {split} pool (ignored)")

print("\nDate filter (a case only cites older cases):")
for split in ["train", "dev"]:
    queries = SPLITS[split]["queries"]
    lost = sum(len(q["all_gold"]) - len(q["gold"]) for q in queries)
    total = sum(len(q["all_gold"]) for q in queries)
    removed = np.mean([(len(q["banned"]) - (q["self"] is not None)) / len(SPLITS[split]["pool"]) for q in queries])
    print(f"   {split:5s}: removes {removed:.1%} of the pool per query | relevant cases lost {lost}/{total} ({lost / max(total, 1):.1%})")

for split in ["train", "dev"]:
    assert sum(len(q["gold"]) > 0 for q in SPLITS[split]["queries"]), f"no {split} query has a reachable relevant case"

split   queries   pool  relevant/query       words/case  with a year  oracle F1@7
train       827   4320   6.42 (max 52)    4508 /   3632        98.3%       0.6386
dev         118   1023   8.25 (max 92)    4620 /   4036        99.3%       0.6181
test        237   1727       (private)    4653 /   3947        98.1%            -

Date filter (a case only cites older cases):
   train: removes 43.4% of the pool per query | relevant cases lost 408/5308 (7.7%)
   dev  : removes 42.5% of the pool per query | relevant cases lost 71/973 (7.3%)


### CELL 6 · Query citation contexts + candidate passages (text)

**In simple words:** Where a query cites a case, the text says `<CITATION>`. The sentence around it describes the cited case, so every `<CITATION>` sentence (± one sentence) becomes a small search query. Candidates are cut into 200-word passages.

**How it works:** ≈ 6–7 contexts per query (at most 48); at most 16 passages per candidate, spread over the case.

In [6]:
# ============================================================
# CELL 6 — QUERY CITATION CONTEXTS + CANDIDATE PASSAGES (text)
#
#   query: every sentence with a <CITATION> (+ CITATION_WINDOW sentences before / after)
#          is one context; a query without any <CITATION> is read as passages instead
#   candidate: passages of PASSAGE_WORDS words; a long case keeps MAX_CHUNKS of them,
#          evenly spread over the case
# Masked parts (<CITATION>, <NAME>, <ORG>) become ordinary words (CELL 3: clean).
# ============================================================

def passages_of(sentences, n_max):
    words = clean(" ".join(sentences)).split() or ["."]
    starts = list(range(0, len(words), PASSAGE_WORDS))
    if len(starts) > n_max:
        starts = [starts[int(round(x))] for x in np.linspace(0, len(starts) - 1, n_max)]
    return [" ".join(words[s:s + PASSAGE_WORDS]) for s in starts]


def contexts_of(sentences):
    """-> list of (text before the citation sentence, citation sentence, text after)."""
    cited = [i for i, s in enumerate(sentences) if "<CITATION>" in s]
    if len(cited) > MAX_CONTEXTS:
        cited = [cited[int(round(x))] for x in np.linspace(0, len(cited) - 1, MAX_CONTEXTS)]
    w = CITATION_WINDOW
    return [(clean(" ".join(sentences[max(0, i - w):i])), clean(sentences[i]), clean(" ".join(sentences[i + 1:i + 1 + w])))
            for i in cited]


for split, S in SPLITS.items():
    for q in S["queries"]:
        q["contexts"] = contexts_of(q["sentences"]) or [("", p, "") for p in passages_of(q["sentences"], MAX_CHUNKS)]
    for c in S["pool"]:
        c["passages"] = passages_of(c["sentences"], MAX_CHUNKS)

for split, S in SPLITS.items():
    no_citation = sum("<CITATION>" not in " ".join(q["sentences"]) for q in S["queries"])
    print(f"{split:6s}: {np.mean([len(q['contexts']) for q in S['queries']]):.1f} contexts per query "
          f"({no_citation} queries without <CITATION> -> passages) | "
          f"{np.mean([len(c['passages']) for c in S['pool']]):.1f} passages per candidate")

example = SPLITS["train"]["queries"][0]["contexts"][0]
print("\nfirst train context (citation sentence only):", example[1][:200])

train : 5.7 contexts per query (1 queries without <CITATION> -> passages) | 12.9 passages per candidate
dev   : 6.9 contexts per query (0 queries without <CITATION> -> passages) | 13.2 passages per candidate
test  : 6.3 contexts per query (0 queries without <CITATION> -> passages) | 13.2 passages per candidate



### CELL 7 · BM25 scores (own implementation) with the same structure as the model

**In simple words:** Classic word-overlap search (**BM25**): a candidate scores high when one of its passages shares rare words with a citation context.

**How it works:** Own implementation (k1 = 1.2, b = 0.75, idf over the split's passages). Per context the scores are standardized over the pool (z-score); a candidate's score = its best passage, a query's score = its best context. BM25 also picks the hard negatives used in training.

In [7]:
# ============================================================
# CELL 7 — BM25 SCORES (own implementation) WITH THE SAME STRUCTURE AS THE MODEL
#
#   score(context, candidate) = BM25 of the context against the candidate's best passage
#                               (idf from the passages of the split's pool)
#   standardized over the pool (z-score per context), then max over the query's contexts
# Used for (1) the hard negatives + which passages a candidate shows in training,
# (2) the signals of the re-ranker (CELL 10), (3) reference scores in the reports.
# Also kept for the re-ranker: the 2nd-best context of every candidate, the share of the query's
# contexts that have it in their top 3, and BM25 of the whole query vs the whole candidate.
# ============================================================

WORD_PATTERN = re.compile(r"[a-z0-9]+")
STOPWORDS = set("""the of and to in a is that be by for or as on with was it this which an are from at
not has have had were been any such shall under said its no may their than also other he she his her
they them there these those if but so all into upon who whom what when where would could should will
citation name org""".split())


def bm25_words(text):
    return [w for w in WORD_PATTERN.findall(text.lower()) if len(w) > 1 and w not in STOPWORDS]


def bm25_matrix(texts, k1=BM25_K1, b=BM25_B):
    """Texts -> (BM25 weight matrix [n_texts, V] sparse, vocabulary)."""

    vocab, rows, cols, vals, lengths = {}, [], [], [], []

    for i, text in enumerate(texts):
        counts = Counter(bm25_words(text))
        lengths.append(sum(counts.values()))
        for w, c in counts.items():
            rows.append(i)
            cols.append(vocab.setdefault(w, len(vocab)))
            vals.append(c)

    n, V = len(texts), len(vocab)
    tf = sparse.csr_matrix((np.array(vals, dtype=np.float32), (rows, cols)), shape=(n, V))
    lengths = np.maximum(np.array(lengths, dtype=np.float32), 1.0)

    df = np.bincount(tf.indices, minlength=V)
    idf = np.log(1.0 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)

    tf = tf.tocoo()
    weight = idf[tf.col] * tf.data * (k1 + 1) / (tf.data + k1 * (1 - b + b * lengths[tf.row] / lengths.mean()))

    return sparse.csr_matrix((weight, (tf.row, tf.col)), shape=(n, V)), vocab


def query_matrix(texts, vocab):
    rows, cols, vals = [], [], []
    for i, text in enumerate(texts):
        for w, c in Counter(bm25_words(text)).items():
            if w in vocab:
                rows.append(i)
                cols.append(vocab[w])
                vals.append(np.log1p(c))
    return sparse.csr_matrix((np.array(vals, dtype=np.float32), (rows, cols)), shape=(len(texts), len(vocab)))


def zscore(scores):
    """Standardize every row over the pool (so different contexts / scores can be compared)."""
    finite = np.where(np.isfinite(scores), scores, np.nan)
    mean = np.nanmean(finite, axis=1, keepdims=True)
    std = np.nanstd(finite, axis=1, keepdims=True) + 1e-6
    return np.where(np.isfinite(scores), (scores - mean) / std, -np.inf).astype(np.float32)


def max_over_owner(scores, owner, n_owners):
    """[rows, n_items] -> [rows, n_owners]: best item of every owner (items sorted by owner)."""
    bounds = np.searchsorted(owner, np.arange(n_owners + 1))
    return np.maximum.reduceat(scores, bounds[:-1], axis=1)


def max_over_contexts(context_scores, context_owner, n_queries):
    """[n_contexts, n_pool] -> [n_queries, n_pool]: z-score every context, then the best context of a query."""
    Z = zscore(context_scores)
    out = np.full((n_queries, Z.shape[1]), -np.inf, dtype=np.float32)
    for r, i in enumerate(context_owner):
        np.maximum(out[i], Z[r], out=out[i])
    return out


def context_text(context):
    return " ".join(part for part in context if part)


def bm25_split(split, train_choices=False, block=200):
    """Query-level BM25 scores [n_queries, n_pool] (+ S["bm25_2nd"], S["bm25_top3"] for the re-ranker);
    train_choices=True also stores, for every train query, its hard negatives and the best passages
    of its relevant / hard candidates."""

    S = SPLITS[split]
    passages = [p for c in S["pool"] for p in c["passages"]]
    owner = np.array([j for j, c in enumerate(S["pool"]) for _ in c["passages"]])
    W, vocab = bm25_matrix(passages)
    shape = (len(S["queries"]), len(S["pool"]))
    scores = np.full(shape, -np.inf, dtype=np.float32)
    S["bm25_2nd"] = np.zeros(shape, dtype=np.float32)       # 2nd-best context of every candidate
    S["bm25_top3"] = np.zeros(shape, dtype=np.float32)      # share of the query's contexts with it in their top 3

    for start in range(0, len(S["queries"]), block):
        queries = S["queries"][start:start + block]
        texts = [context_text(c) for q in queries for c in q["contexts"]]
        ctx_owner = np.array([i for i, q in enumerate(queries) for _ in q["contexts"]])

        passage_scores = (query_matrix(texts, vocab) @ W.T).toarray()                    # [contexts, passages]
        Z = zscore(max_over_owner(passage_scores, owner, len(S["pool"])))                 # [contexts, pool]

        for i in range(len(queries)):
            z = np.sort(Z[ctx_owner == i], axis=0)                                        # contexts sorted per candidate
            scores[start + i] = z[-1]                                                     # = max_over_contexts
            S["bm25_2nd"][start + i] = z[-2] if len(z) > 1 else z[-1] - 3.0
            top3 = np.argpartition(-Z[ctx_owner == i], 2, axis=1)[:, :3]
            S["bm25_top3"][start + i] = np.bincount(top3.ravel(), minlength=len(S["pool"])) / len(z)

        if train_choices:
            for i, q in enumerate(queries):
                best_passage = passage_scores[ctx_owner == i].max(axis=0)                   # [passages]
                row = scores[start + i].copy()
                row[q["banned"]] = -np.inf
                row[q["all_gold"]] = -np.inf
                q["hard"] = [int(k) for k in np.argsort(-row)[:HARD_POOL] if np.isfinite(row[k])]
                q["show"] = {}
                for c in set(q["gold"].tolist()) | set(q["hard"]):
                    first = np.searchsorted(owner, c)
                    mine = best_passage[first:first + len(S["pool"][c]["passages"])]
                    q["show"][c] = sorted(np.argsort(-mine)[:TRAIN_CHUNKS].tolist())        # passages shown in training

    return scores


def whole_case_bm25(split):
    """Whole query vs whole candidate (no contexts, no passages): a reference, and a re-ranker signal."""
    S = SPLITS[split]
    W, vocab = bm25_matrix([clean(" ".join(c["sentences"])) for c in S["pool"]])
    Q = query_matrix([clean(" ".join(q["sentences"])) for q in S["queries"]], vocab)
    return (Q @ W.T).toarray().astype(np.float32)


t0 = time.time()

for split, S in SPLITS.items():
    S["bm25"] = bm25_split(split, train_choices=(split == "train"))
    S["bm25_whole"] = whole_case_bm25(split)                                        # reference (CELL 9) + re-ranker (CELL 10)
    print(f"{split:6s}: BM25 {S['bm25'].shape[0]} x {S['bm25'].shape[1]} | {time.time() - t0:.0f}s", flush=True)

for split, S in SPLITS.items():
    for d in S["queries"] + S["pool"]:
        del d["sentences"]                                # free memory (contexts / passages are kept)

train : BM25 827 x 4320 | 44s
dev   : BM25 118 x 1023 | 52s
test  : BM25 237 x 1727 | 64s


### CELL 8 · Tokenize: one chunk per citation context and per candidate passage

**In simple words:** Turn every citation context and every passage into token ids (at most 512 each).

**How it works:** A long context keeps its `<CITATION>` sentence and as much text around it as fits.

In [8]:
# ============================================================
# CELL 8 — TOKENIZE: one chunk per citation context and per candidate passage
#   a long context keeps its citation sentence and as much of the text around it as fits
# ============================================================

def context_chunk(context):
    before, sentence, after = (encode_fast(" " + part.strip()) if part.strip() else [] for part in context)
    sentence = sentence[:CHUNK_LEN]
    room = CHUNK_LEN - len(sentence)
    take_before = min(len(before), room // 2 if len(after) > room // 2 else room - len(after))
    ids = before[len(before) - take_before:] + sentence + after[:room - take_before]
    return np.array(ids or encode_fast("."), dtype=np.int32)


t0 = time.time()

for split, S in SPLITS.items():
    for q in S["queries"]:
        q["chunks"] = [context_chunk(c) for c in q["contexts"]]
    for c in S["pool"]:
        c["chunks"] = [np.array(head_tokens(p, CHUNK_LEN) or encode_fast("."), dtype=np.int32) for p in c["passages"]]
        del c["passages"]
    print(f"{split:6s}: tokenized | {time.time() - t0:.0f}s", flush=True)

for split, S in SPLITS.items():
    ctx = [len(x) for q in S["queries"] for x in q["chunks"]]
    pas = [len(x) for c in S["pool"] for x in c["chunks"]]
    print(f"{split:6s}: context {np.median(ctx):.0f} tokens (median) | passage {np.median(pas):.0f} tokens, "
          f"{np.mean([p >= CHUNK_LEN for p in pas]):.1%} cut at {CHUNK_LEN}")

sample_ids = np.concatenate([x for c in SPLITS["train"]["pool"][:300] for x in c["chunks"]])
print(f"\nwhitespace tokens: {np.isin(sample_ids, SPACE_TOKEN_IDS).mean():.1%} of all tokens "
      f"({'not pooled' if POOL_SKIP_SPACES else 'pooled'})")

train : tokenized | 14s
dev   : tokenized | 17s
test  : tokenized | 22s
train : context 512 tokens (median) | passage 479 tokens, 10.0% cut at 512
dev   : context 512 tokens (median) | passage 479 tokens, 10.8% cut at 512
test  : context 512 tokens (median) | passage 479 tokens, 11.1% cut at 512

whitespace tokens: 41.1% of all tokens (not pooled)


### CELL 9 · Ranking + micro-F1@7 (same rule as the leaderboard)

**In simple words:** How the leaderboard scores: exactly 7 candidates per query; precision = correct / 7, recall = correct / all relevant, and F1 combines the two over all queries (micro).

**How it works:** Recall counts every relevant case, also those the date filter removed. The table shows how much each idea helps BM25.

In [9]:
# ============================================================
# CELL 9 — RANKING + MICRO-F1@7 (same rule as the leaderboard)
#
#   micro-precision@7 = all correct among the 7 predictions / (7 x queries)
#   micro-recall@7    = all correct / all relevant cases (ALL of them, also those the date filter removed)
#   micro-F1@7        = harmonic mean of the two
# ============================================================

def mask_banned(scores, queries):
    """The query itself and (date filter) newer cases are never ranked."""
    scores = scores.copy()
    for i, q in enumerate(queries):
        scores[i, q["banned"]] = -np.inf
    return scores


def top_k(scores, k=TOP_K):
    """[n_queries, n_pool] -> [n_queries, k] pool indices, best first."""
    k = min(k, scores.shape[1])
    part = np.argpartition(-scores, k - 1, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(scores, part, axis=1), axis=1)
    return np.take_along_axis(part, order, axis=1)


def f1_at_k(ranked, queries):
    """ranked [n_queries, k] -> dict with micro P / R / F1 @k."""

    tp = sum(len(set(r.tolist()) & set(q["all_gold"].tolist())) for r, q in zip(ranked, queries))
    n_pred = ranked.shape[0] * ranked.shape[1]
    n_gold = sum(len(q["all_gold"]) for q in queries)

    p = tp / max(n_pred, 1)
    r = tp / max(n_gold, 1)

    return {"precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0, "hits": tp}


def score_f1(scores, queries):
    return f1_at_k(top_k(mask_banned(scores, queries)), queries)


def fuse(model_scores, other_scores, weight):
    """weight * model + (1 - weight) * other (the re-ranker), both standardized per query.
    Only the candidates the other score ranks (the re-ranker: the BM25 top 100) can be predicted."""
    other = zscore(other_scores)
    reachable = np.isfinite(other)
    fused = weight * zscore(model_scores) + (1 - weight) * np.where(reachable, other, 0.0)
    return np.where(reachable, fused, -np.inf).astype(np.float32)


def best_fusion(model_scores, other_scores, queries):
    """Weight of the model score with the best micro-F1@7 on these queries -> (weight, results of every weight)."""
    results = {w: score_f1(fuse(model_scores, other_scores, w), queries) for w in FUSION_WEIGHTS}
    best = max(FUSION_WEIGHTS, key=lambda w: (round(results[w]["f1"], 6), w))
    return best, results


def recall_at(scores, queries, k):
    """Share of the relevant cases in the top k."""
    ranked = top_k(mask_banned(scores, queries), k)
    found = sum(len(set(r.tolist()) & set(q["all_gold"].tolist())) for r, q in zip(ranked, queries))
    return found / max(sum(len(q["all_gold"]) for q in queries), 1)


# check on a tiny example: query 0 has gold {0, 1}, query 1 has gold {2}
_queries = [{"all_gold": np.array([0, 1]), "banned": np.array([], dtype=np.int64)},
            {"all_gold": np.array([2]), "banned": np.array([], dtype=np.int64)}]
_m = f1_at_k(top_k(np.array([[0.9, 0.1, 0.8], [0.1, 0.2, 0.3]]), 2), _queries)
assert _m["hits"] == 2 and abs(_m["precision"] - 0.5) < 1e-9 and abs(_m["recall"] - 2 / 3) < 1e-9
print("✅ micro-F1@k check passed")

def self_only(queries):
    """The same queries without the date filter (only the query itself is never ranked)."""
    return [{**q, "banned": np.array([] if q["self"] is None else [q["self"]], dtype=np.int64)} for q in queries]


print(f"\n{'BM25 reference scores':50s} {'train F1@7':>11s} {'dev F1@7':>9s}")
for name, key, date_filter in [("whole query vs whole candidate", "bm25_whole", False),
                               ("+ date filter", "bm25_whole", True),
                               ("citation contexts vs passages + date filter (used)", "bm25", True)]:
    row = [score_f1(SPLITS[s][key], SPLITS[s]["queries"] if date_filter else self_only(SPLITS[s]["queries"]))["f1"]
           for s in ["train", "dev"]]
    print(f"{name:50s} {row[0]:11.4f} {row[1]:9.4f}")

for split in ["train", "dev"]:
    print(f"{split}: oracle F1@7 {oracle_f1(SPLITS[split]['queries']):.4f} | BM25 has "
          f"{recall_at(SPLITS[split]['bm25'], SPLITS[split]['queries'], HARD_POOL):.1%} of the relevant cases in its top {HARD_POOL}, "
          f"{recall_at(SPLITS[split]['bm25'], SPLITS[split]['queries'], 100):.1%} in its top 100")

✅ micro-F1@k check passed

BM25 reference scores                               train F1@7  dev F1@7
whole query vs whole candidate                          0.1415    0.1556
+ date filter                                           0.1727    0.2034
citation contexts vs passages + date filter (used)      0.3893    0.4647
train: oracle F1@7 0.6386 | BM25 has 64.9% of the relevant cases in its top 30, 75.5% in its top 100
dev: oracle F1@7 0.6181 | BM25 has 65.5% of the relevant cases in its top 30, 77.9% in its top 100


### CELL 10 · Learned re-ranker (logistic regression, fitted on train queries only)

**In simple words:** A small logistic regression learns from the **train** queries how to weigh 9 BM25-based clues — best context, second-best context, whole-case match, years between the cases, Supreme Court judgment, length, …. It lifts BM25 from 0.465 to 0.473 on dev.

**How it works:** It re-scores the BM25 top 100 of every query; the printed weights show which clues matter (whole-case BM25 and the number of contexts most). Fitted on train queries only, so dev and test stay unseen.

In [10]:
# ============================================================
# CELL 10 — LEARNED RE-RANKER (logistic regression, fitted on TRAIN queries only)
#
# BM25 alone ranks by the best citation context only. The re-ranker weighs 9 signals of every
# candidate in the BM25 top RERANK_TOP of a query (all from the texts, no labels of dev / test):
#   BM25 best context, BM25 2nd-best context, share of the query's contexts with it in their top 3,
#   BM25 best minus the query's 7th best, BM25 whole case vs whole case, log(1 + years between the
#   cases), Supreme Court judgment (its header), log(words of the candidate), log(contexts of the query)
# label = the candidate is relevant to the train query. Checked on the real data: 5-fold on train
# queries 0.404 vs BM25 0.390 (better in every fold), dev 0.473 vs 0.465.
# The model (from CELL 11 on) is then fused with this score.
# ============================================================

from sklearn.linear_model import LogisticRegression

FEATURE_NAMES = ["BM25 best context", "BM25 2nd-best context", "share of contexts with it in their top 3",
                 "BM25 best - the query's 7th best", "BM25 whole case", "log(1 + years between the cases)",
                 "Supreme Court judgment", "log(words of the candidate)", "log(contexts of the query)"]


def ranker_features(split):
    """-> features [rows, 9], rows = (query index, candidate index) of the BM25 top RERANK_TOP of every query."""

    S = SPLITS[split]
    best = mask_banned(S["bm25"], S["queries"])
    whole = zscore(S["bm25_whole"])
    X, rows = [], []

    for i, q in enumerate(S["queries"]):
        candidates = [c for c in np.argsort(-best[i])[:RERANK_TOP] if np.isfinite(best[i, c])]
        seventh = best[i, candidates[min(TOP_K, len(candidates)) - 1]]

        for c in candidates:
            d = S["pool"][c]
            years = np.log1p(max(q["year"] - d["year"], 0)) if q["year"] and d["year"] else -1.0
            X.append([best[i, c], S["bm25_2nd"][i, c], S["bm25_top3"][i, c], best[i, c] - seventh, whole[i, c],
                      years, float(d["supreme_court"]), np.log1p(d["words"]), np.log1p(len(q["contexts"]))])
            rows.append((i, c))

    return np.array(X, dtype=np.float32), rows


# ------------------------------------------------------------
# Fit on TRAIN queries (features standardized with the train mean / std)
# ------------------------------------------------------------

X_train, rows_train = ranker_features("train")
relevant = [set(q["all_gold"].tolist()) for q in SPLITS["train"]["queries"]]
y_train = np.array([c in relevant[i] for i, c in rows_train])

FEATURE_MEAN, FEATURE_STD = X_train.mean(axis=0), X_train.std(axis=0) + 1e-6
ranker = LogisticRegression(C=RERANK_C, max_iter=3000)
ranker.fit((X_train - FEATURE_MEAN) / FEATURE_STD, y_train)


def ranker_scores(split):
    """[n_queries, n_pool] re-ranker score (log-odds); -inf outside the BM25 top RERANK_TOP."""
    X, rows = ranker_features(split)
    values = ranker.decision_function((X - FEATURE_MEAN) / FEATURE_STD)
    R = np.full(SPLITS[split]["bm25"].shape, -np.inf, dtype=np.float32)
    for (i, c), v in zip(rows, values):
        R[i, c] = v
    return R


for split, S in SPLITS.items():
    S["ranker"] = ranker_scores(split)
    assert all(np.isfinite(row).sum() >= TOP_K for row in mask_banned(S["ranker"], S["queries"])), split

print(f"fitted on {len(y_train)} train (query, candidate) pairs, {y_train.sum()} relevant\n")
print(f"{'signal':42s} {'weight (standardized)':>22s}")
for name, w in zip(FEATURE_NAMES, ranker.coef_[0]):
    print(f"{name:42s} {w:+22.3f}")

print(f"\n{'':20s} {'train F1@7':>11s} {'dev F1@7':>9s}")
for name, key in [("BM25", "bm25"), ("re-ranker", "ranker")]:
    row = [score_f1(SPLITS[s][key], SPLITS[s]["queries"])["f1"] for s in ["train", "dev"]]
    print(f"{name:20s} {row[0]:11.4f} {row[1]:9.4f}")
print("(train F1 of the re-ranker is in-sample: it was fitted on these queries)")

fitted on 82690 train (query, candidate) pairs, 4008 relevant

signal                                      weight (standardized)
BM25 best context                                          +0.399
BM25 2nd-best context                                      +0.278
share of contexts with it in their top 3                   +0.157
BM25 best - the query's 7th best                           +0.469
BM25 whole case                                            +0.637
log(1 + years between the cases)                           +0.016
Supreme Court judgment                                     +0.228
log(words of the candidate)                                -0.633
log(contexts of the query)                                 +0.809

                      train F1@7  dev F1@7
BM25                      0.3893    0.4647
re-ranker                 0.4035    0.4725
(train F1 of the re-ranker is in-sample: it was fitted on these queries)


### CELL 11 · Model

**In simple words:** The Phase-1 model as a **retriever**: every context or passage becomes one vector, and texts about the same legal point get vectors that point the same way (high cosine similarity).

**How it works:** token vectors = learned mix of all layers → standardized → mean over the non-space tokens → projection (starts as the identity) → unit vector. Adaptation: LoRMA, rank 16, α 32, on qkv / out_proj / gate / up / down of all 12 blocks (3.05M weights, 2.71%) + retriever head (0.59M).

In [11]:
# ============================================================
# CELL 11 — MODEL
#
#   every chunk -> Architecture-1 decoder (Phase-1 weights FROZEN) + LoRMA on every linear layer
#   -> token vectors of ALL layers -> retriever head (below)
#   retriever head (per chunk): layer mix -> standardize -> mean over tokens -> projection -> unit vector
#
# Layer names match the Phase-1 notebook, so the weights load directly.
# ============================================================

# ============================================================
# Architecture-1 DECODER (causal, RoPE)
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * rms * self.weight).to(x.dtype)


def build_rope_cache(block_size, head_dim, theta):

    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(block_size).float(), inv_freq)

    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):

    x1, x2 = x.float().chunk(2, dim=-1)

    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).to(x.dtype)


class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        d_model = config["d_model"]

        self.n_heads = config["n_heads"]
        self.head_dim = d_model // self.n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

    def forward(self, x, cos, sin):

        B, T, C = x.shape

        q, k, v = self.qkv(x).split(C, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(self.q_norm(q), cos, sin)
        k = apply_rope(self.k_norm(k), cos, sin)

        # causal, as in pretraining. Padding is at the END, so real tokens never see it.
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        return self.out_proj(y.transpose(1, 2).contiguous().view(B, T, C))


class SwiGLU(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.gate = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.up = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.down = nn.Linear(config["d_ff"], config["d_model"], bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class TransformerBlock(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.norm1 = RMSNorm(config["d_model"])
        self.attn = CausalSelfAttention(config)
        self.norm2 = RMSNorm(config["d_model"])
        self.mlp = SwiGLU(config)

    def forward(self, x, cos, sin):

        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.mlp(self.norm2(x))

        return x


class DecoderBackbone(nn.Module):
    """Architecture-1 without the LM head: tokens -> hidden vectors."""

    def __init__(self, config):
        super().__init__()

        self.token_embedding = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config["n_layers"])])
        self.norm_f = RMSNorm(config["d_model"])

        cos, sin = build_rope_cache(config["block_size"], config["d_model"] // config["n_heads"], config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, input_ids, mask=None, all_layers=False):
        """`mask` is not needed: attention is causal and padding is at the end.
        all_layers=False: final hidden vectors (as in Phase 1).
        all_layers=True : list of the embeddings + the output of every block (before any norm)."""

        T = input_ids.size(1)

        cos, sin = self.rope_cos[:T], self.rope_sin[:T]

        x = self.token_embedding(input_ids)
        states = [x]

        for block in self.blocks:
            x = block(x, cos, sin)
            states.append(x)

        if all_layers:
            return states

        return self.norm_f(x)


def backbone_blocks(backbone):
    """The transformer blocks of the Phase-1 model, bottom -> top."""

    return list(backbone.blocks)


# ------------------------------------------------------------
# LoRMA (multiplicative low-rank adaptation)
#     W' x = (I + alpha/r * B A) W x = h + alpha/r * B (A h),   h = W x
# A: out_features -> r,  B: r -> out_features,  B = 0 at the start (W' = W).
# A bias (only Architecture-2 has them) is kept as it is: y = W' x + b
# ------------------------------------------------------------

class LoRMALinear(nn.Module):

    def __init__(self, base, rank, alpha, dropout):
        super().__init__()

        self.base = base                                      # frozen Phase-1 layer
        out_features = base.out_features

        self.lorma_A = nn.Linear(out_features, rank, bias=False)
        self.lorma_B = nn.Linear(rank, out_features, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale = alpha / rank

        nn.init.kaiming_uniform_(self.lorma_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B.weight)                   # starts as the identity -> W' = W

    def forward(self, x):

        h = F.linear(x, self.base.weight)                                       # W x
        h = h + self.scale * self.lorma_B(self.lorma_A(self.dropout(h)))       # (I + alpha/r * B A) W x

        if self.base.bias is not None:
            h = h + self.base.bias.to(h.dtype)                                  # + b (frozen)

        return h


def add_peft(backbone):
    """Freeze every Phase-1 weight, then wrap the target linear layers of every block with LoRMA."""

    for p in backbone.parameters():
        p.requires_grad = False

    for block in backbone.blocks:
        for target in LORMA_TARGETS:                          # e.g. "attn.qkv"

            parent_name, name = target.rsplit(".", 1)
            parent = block.get_submodule(parent_name)

            setattr(parent, name, LoRMALinear(getattr(parent, name), LORMA_RANK, LORMA_ALPHA, LORMA_DROPOUT))


def backbone_groups(model):
    """(parameters, learning rate, weight decay) of the trained Phase-1 part."""
    return [(model.backbone_parameters(), LR_PEFT, 0.0)]


def check_trainable(backbone):
    names = [name for name, p in backbone.named_parameters() if p.requires_grad]
    assert names and all("lorma_" in name for name in names), "only LoRMA matrices may train in the Phase-1 model"


# ------------------------------------------------------------
# Check: LoRMA really computes (I + alpha/r * B A) W x
# ------------------------------------------------------------

_layer = LoRMALinear(nn.Linear(8, 6, bias=False), rank=2, alpha=4, dropout=0.0)
nn.init.normal_(_layer.lorma_B.weight)
_x = torch.randn(3, 8)
_merged = (torch.eye(6) + _layer.scale * _layer.lorma_B.weight @ _layer.lorma_A.weight) @ _layer.base.weight
assert torch.allclose(_layer(_x), _x @ _merged.T, atol=1e-5)
print("✅ LoRMA layer = (I + alpha/r * B A) W x")

# ============================================================
# Chunks -> padded token ids, and which tokens are pooled
# ============================================================

SPACE_IDS = torch.tensor(SPACE_TOKEN_IDS if POOL_SKIP_SPACES and SPACE_TOKEN_IDS else [-1], dtype=torch.long, device=DEVICE)


def pad_chunks(chunks):
    """List of token-id arrays -> ids [n, T], mask [n, T] (True = real token), on the GPU."""

    max_len = max(len(c) for c in chunks)
    ids = torch.full((len(chunks), max_len), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(chunks), max_len), dtype=torch.bool)

    for row, c in enumerate(chunks):
        ids[row, :len(c)] = torch.from_numpy(c.astype(np.int64))
        mask[row, :len(c)] = True

    return ids.to(DEVICE), mask.to(DEVICE)


def pooling_mask(ids, mask):
    """Tokens that are pooled: real tokens, not whitespace, not the first POOL_SKIP_TOKENS of a chunk."""

    pool = mask & ~torch.isin(ids, SPACE_IDS)

    long_chunk = pool.sum(dim=1) > 2 * POOL_SKIP_TOKENS
    pool[long_chunk, :POOL_SKIP_TOKENS] = False

    empty = ~pool.any(dim=1)                   # a chunk with nothing left keeps all its real tokens
    pool[empty] = mask[empty]

    return pool


# ============================================================
# Retriever: chunk (citation context or passage) -> unit vector
# ============================================================

class Retriever(nn.Module):

    def __init__(self, backbone, d_model, n_states):
        super().__init__()

        self.backbone = backbone

        # layer mix: token vector = sum_l w_l * LayerNorm(h_l), w = softmax(layer_logits) (starts uniform)
        self.layer_logits = nn.Parameter(torch.zeros(n_states))

        # token standardization (x - mean) / std, statistics of the Phase-1 model (CELL 15), fixed
        self.register_buffer("token_mean", torch.zeros(d_model))
        self.register_buffer("token_std", torch.ones(d_model))

        # projection: starts as the identity, so the start = standardized mean pooling of the Phase-1 model
        self.projection = nn.Linear(d_model, d_model)
        nn.init.eye_(self.projection.weight)
        nn.init.zeros_(self.projection.bias)

    def backbone_parameters(self):
        return [p for p in self.backbone.parameters() if p.requires_grad]

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("backbone.")]

    def token_vectors_mean(self, chunks, standardize=True):
        """List of token-id arrays -> [n, D] float32: mean of the pooled (mixed, standardized) token vectors."""

        ids, mask = pad_chunks(chunks)
        pool = pooling_mask(ids, mask)

        with autocast():
            states = self.backbone(ids, mask, all_layers=True)

        weights = torch.softmax(self.layer_logits.float(), dim=0)
        mixed = 0.0
        for w, h in zip(weights, states):
            mixed = mixed + w * F.layer_norm(h.float(), h.shape[-1:])

        if standardize:
            mixed = (mixed - self.token_mean) / self.token_std

        m = pool.unsqueeze(-1).float()
        return (mixed * m).sum(dim=1) / m.sum(dim=1)

    def chunk_vectors(self, chunks):
        """List of token-id arrays -> [n, D] unit vectors."""
        return F.normalize(self.projection(self.token_vectors_mean(chunks)), dim=-1)


def select_chunks(chunks, k):
    """At most k chunks, evenly spread over the case."""
    if len(chunks) <= k:
        return chunks
    return [chunks[i] for i in np.linspace(0, len(chunks) - 1, k).round().astype(int)]


@torch.no_grad()
def encode_chunks(model, chunks):
    """Token-id arrays -> [n, D] float32 unit vectors on the CPU (no gradients; similar lengths batched together)."""

    model.eval()

    order = np.argsort([len(c) for c in chunks])
    out = np.zeros((len(chunks), model.projection.out_features), dtype=np.float32)

    for k in range(0, len(chunks), EVAL_CHUNKS):
        rows = order[k:k + EVAL_CHUNKS]
        out[rows] = model.chunk_vectors([chunks[i] for i in rows]).float().cpu().numpy()

    model.train()
    return out

✅ LoRMA layer = (I + alpha/r * B A) W x


### CELL 12 · Build the model from the Phase-1 Architecture-1 checkpoint

**In simple words:** Create the network, load the trained Phase-1 weights into it, and add the adaptation (the part that will learn).

**How it works:** The layer names match the Phase-1 notebook, so the checkpoint loads directly; if it is not at the configured path, the notebook searches `/kaggle/input`. The printed counts show how many weights train. Here: LoRMA, rank 16, α 32, on qkv / out_proj / gate / up / down of all 12 blocks (3.05M weights, 2.71%) + retriever head (0.59M).

In [12]:
# ============================================================
# CELL 12 — BUILD THE MODEL FROM THE PHASE-1 ARCHITECTURE-1 CHECKPOINT
# ============================================================

def is_architecture1(path):

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        return isinstance(ckpt, dict) and "rope_theta" in ckpt.get("model_config", {})
    except Exception:
        return False


if not CHECKPOINT_PATH.exists():

    found = [p for p in sorted(Path("/kaggle/input").rglob("best_model.pt")) if is_architecture1(p)]
    assert found, f"{CHECKPOINT_PATH} not found and no Architecture-1 checkpoint under /kaggle/input"

    CHECKPOINT_PATH = found[0]
    print("Checkpoint found at:", CHECKPOINT_PATH)


phase1 = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
MODEL_CONFIG = dict(phase1["model_config"])

# weights without the LM head (tied to the embedding, not needed here)
MODEL_WEIGHTS = {
    key.replace("_orig_mod.", ""): value
    for key, value in phase1["model_state_dict"].items()
    if not key.replace("_orig_mod.", "").startswith("lm_head.")
}

del phase1

D_MODEL = MODEL_CONFIG["d_model"]

print("Phase-1 config:", MODEL_CONFIG)

assert MODEL_CONFIG["vocab_size"] == VOCAB_SIZE, "checkpoint was trained with a different tokenizer"
assert CHUNK_LEN <= MODEL_CONFIG["block_size"], f"CHUNK_LEN must be <= {MODEL_CONFIG['block_size']}"


def build_model():
    """Phase-1 model + LoRMA + retriever head."""

    backbone = DecoderBackbone(MODEL_CONFIG)
    backbone.load_state_dict(MODEL_WEIGHTS, strict=True)          # every name and shape must match

    test_ids = torch.randint(4, VOCAB_SIZE, (2, 24))
    test_mask = torch.ones(2, 24, dtype=torch.bool)

    backbone.eval()
    with torch.no_grad():
        before = backbone(test_ids, test_mask)
        add_peft(backbone)
        after = backbone(test_ids, test_mask)
        states = backbone(test_ids, test_mask, all_layers=True)
    backbone.train()

    assert torch.allclose(before, after, atol=1e-4), "LoRMA changed the model at the start"
    assert len(states) == len(backbone_blocks(backbone)) + 1
    assert torch.allclose(backbone.norm_f(states[-1]), after, atol=1e-4)
    check_trainable(backbone)

    return Retriever(backbone, D_MODEL, len(backbone_blocks(backbone)) + 1).to(DEVICE)


for name in ["model", "optimizer", "scaler"]:
    globals().pop(name, None)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

model = build_model()

N_BLOCKS = len(backbone_blocks(model.backbone))

n_total = sum(p.numel() for p in model.parameters())
n_backbone = sum(p.numel() for p in model.backbone.parameters())
n_trained = sum(p.numel() for p in model.backbone_parameters())
n_head = sum(p.numel() for p in model.head_parameters())

print(f"\nPhase-1 model             : {n_backbone:,} ({N_BLOCKS} blocks)")
print(f"trained in it (LoRMA): {n_trained:,}  ({n_trained / n_backbone:.2%})")
print(f"head (trainable)          : {n_head:,}")

Phase-1 config: {'vocab_size': 32004, 'block_size': 2048, 'd_model': 768, 'n_layers': 12, 'n_heads': 12, 'd_ff': 2048, 'rope_theta': 10000.0}

Phase-1 model             : 112,581,888 (12 blocks)
trained in it (LoRMA): 3,047,424  (2.71%)
head (trainable)          : 590,605


### CELL 13 · Model scores + evaluation on a split

**In simple words:** Score every query against every candidate with the model (the same structure as BM25) and evaluate three rankings: the model alone, the re-ranker alone, and the two mixed.

**How it works:** Mix = w · z(model) + (1 − w) · z(re-ranker); w is picked on dev. `dev_report` also saves `results.json`, the dev predictions and the trained weights (`retriever.pt`).

In [13]:
# ============================================================
# CELL 13 — MODEL SCORES + EVALUATION ON A SPLIT
#   score(context, candidate) = best cosine over the candidate's passages, z-scored over the pool,
#   then the best context of the query (the same structure as BM25, CELL 7)
# ============================================================

def model_scores(model, split, n_chunks=MAX_CHUNKS, block=256):
    """[n_queries, n_pool] model scores + encoding time."""

    S = SPLITS[split]
    t0 = time.time()

    contexts = [x for q in S["queries"] for x in q["chunks"]]
    context_owner = [i for i, q in enumerate(S["queries"]) for _ in q["chunks"]]
    passages, passage_owner = [], []
    for j, c in enumerate(S["pool"]):
        chosen = select_chunks(c["chunks"], n_chunks)
        passages.extend(chosen)
        passage_owner.extend([j] * len(chosen))
    passage_owner = np.array(passage_owner)

    Q = encode_chunks(model, contexts)
    P = encode_chunks(model, passages)
    seconds = time.time() - t0

    per_candidate = np.zeros((len(contexts), len(S["pool"])), dtype=np.float32)
    for k in range(0, len(contexts), block):
        per_candidate[k:k + block] = max_over_owner(Q[k:k + block] @ P.T, passage_owner, len(S["pool"]))

    return max_over_contexts(per_candidate, context_owner, len(S["queries"])), seconds


def evaluate_split(model, split, n_chunks=MAX_CHUNKS):
    """Model alone, BM25 alone, re-ranker alone and the best fusion (model + re-ranker) on a labelled split."""

    S = SPLITS[split]
    scores, seconds = model_scores(model, split, n_chunks)
    weight, fused = best_fusion(scores, S["ranker"], S["queries"])

    return {"model": score_f1(scores, S["queries"]), "bm25": score_f1(S["bm25"], S["queries"]),
            "ranker": score_f1(S["ranker"], S["queries"]),
            "fused": fused[weight], "weight": weight, "seconds": seconds, "scores": scores}


def show(name, result):
    print(f"{name:34s} model {result['model']['f1']:.4f} | fused {result['fused']['f1']:.4f} "
          f"(model weight {result['weight']:.1f}) | re-ranker {result['ranker']['f1']:.4f} | "
          f"BM25 {result['bm25']['f1']:.4f} | encoded in {result['seconds']:.0f}s", flush=True)


def dev_report(model, training, sizes, name=RUN_NAME, save=True):
    """Full dev evaluation (MAX_CHUNKS passages per candidate) -> (fusion weight for the test ranking, results)."""

    S = SPLITS["dev"]
    final = evaluate_split(model, "dev", MAX_CHUNKS)
    weight = final["weight"]

    print(f"DEV — {name} ({len(S['queries'])} queries, pool {len(S['pool'])}), {MAX_CHUNKS} passages per candidate\n")
    print(f"{'ranking':40s} {'micro-F1@7':>10s} {'P@7':>8s} {'R@7':>8s}")
    for label, m in [("BM25 alone (reference)", final["bm25"]), ("re-ranker alone (CELL 10)", final["ranker"]),
                     ("the model alone", final["model"]),
                     (f"fused: model {weight:.1f} + re-ranker -> test", final["fused"])]:
        print(f"{label:40s} {m['f1']:10.4f} {m['precision']:8.4f} {m['recall']:8.4f}")
    print(f"{'oracle (7 predictions per query)':40s} {oracle_f1(S['queries']):10.4f}")

    _, all_weights = best_fusion(final["scores"], S["ranker"], S["queries"])
    print("\nmodel weight -> dev micro-F1@7: " + ", ".join(f"{w:.1f}: {m['f1']:.4f}" for w, m in all_weights.items()))

    fused_scores = mask_banned(fuse(final["scores"], S["ranker"], weight), S["queries"])
    ranked = top_k(fused_scores)

    print(f"\nrelevant cases in the top 50: model {recall_at(final['scores'], S['queries'], 50):.1%} | "
          f"BM25 {recall_at(S['bm25'], S['queries'], 50):.1%} | fused {recall_at(fused_scores, S['queries'], 50):.1%}")

    print(f"\n{'relevant cases':>15s} {'queries':>8s} {'fused F1@7':>11s}")
    n_rel = np.array([len(q["all_gold"]) for q in S["queries"]])
    for lo, hi in [(0, 3), (4, 7), (8, 15), (16, 10 ** 6)]:
        rows = np.where((n_rel >= lo) & (n_rel <= hi))[0]
        if len(rows):
            m = f1_at_k(ranked[rows], [S["queries"][i] for i in rows])
            print(f"{f'{lo}-{hi}' if hi < 10 ** 6 else f'{lo}+':>15s} {len(rows):8d} {m['f1']:11.4f}")

    hits = [len(set(r.tolist()) & set(q["all_gold"].tolist())) for r, q in zip(ranked, S["queries"])]
    print("\ncorrect cases among the 7 predictions -> queries:", dict(sorted(Counter(hits).items())))

    n_chunks = sum(len(q["chunks"]) for q in S["queries"]) + sum(len(select_chunks(c["chunks"], MAX_CHUNKS)) for c in S["pool"])
    speed = n_chunks / max(final["seconds"], 1e-6)
    print(f"\nencoding {n_chunks} dev chunks took {final['seconds']:.0f}s ({speed:.0f} chunks/s) | "
          f"index: {n_chunks - sum(len(q['chunks']) for q in S['queries'])} passage vectors x {D_MODEL} floats")

    results = {
        "run": name,
        "dev_bm25": final["bm25"], "dev_ranker": final["ranker"], "dev_model": final["model"], "dev_fused": final["fused"],
        "fusion_weight": weight,
        "fusion_weights": {str(w): m["f1"] for w, m in all_weights.items()},
        "encode_chunks_per_second": speed,
        **sizes, **training,
    }

    if save:
        with open(OUTPUT_DIR / "results.json", "w") as f:
            json.dump(results, f, indent=1, default=float)
        with open(OUTPUT_DIR / "dev_predictions.json", "w") as f:
            json.dump({q["id"]: [S["pool"][k]["id"] for k in r] for q, r in zip(S["queries"], ranked)}, f, indent=1)
        torch.save({"run": name, "fusion_weight": weight, "state": trainable_state(model)}, OUTPUT_DIR / "retriever.pt")
        print(f"\nSaved: {OUTPUT_DIR / 'results.json'} | dev_predictions.json | retriever.pt (trained weights only)")

    return weight, results

### CELL 14 · Contrastive training function (GradCache: big batches with little memory)

**In simple words:** Training teaches the model to score the truly cited case **higher** than look-alike wrong cases (BM25's near misses) and the other candidates in the batch.

**How it works:** InfoNCE loss, temperature 0.05. GradCache: encode all chunks without gradients, compute the loss, then re-encode small groups with gradients — the gradient of a big batch with the memory of a small one. Also: `find_trained` / `load_trained` to reuse an earlier `retriever.pt`.

In [14]:
# ============================================================
# CELL 14 — CONTRASTIVE TRAINING FUNCTION (GradCache: big batches with little memory)
#
# One weight update = BATCH_QUERIES queries, each with 1 relevant candidate and HARD_NEGATIVES
# BM25 hard negatives; every query is scored against ALL candidates of the update:
#   score(query, candidate) = max over its contexts, max over the candidate's passages, of cos / TEMPERATURE
# (InfoNCE; other relevant cases of a query are left out of its denominator).
# A candidate shows TRAIN_CHUNKS passages: its best BM25 passages for that query (CELL 7).
#
# GradCache (Gao et al., 2021): pass 1 encodes every chunk WITHOUT gradients and computes the
# loss and its gradient w.r.t. the chunk vectors; pass 2 re-encodes the chunks in groups WITH
# gradients and back-propagates that gradient. Same gradient as one big batch, but only
# EMBED_GROUP_CHUNKS chunks in memory. (Dropout is 0 here, so both passes compute the same vectors.)
# ============================================================

def make_batch(queries, rng):
    """-> chunks (contexts first, then candidate passages), their owners, targets, blocked mask, candidate ids."""

    pool = SPLITS["train"]["pool"]
    context_chunks, context_owner, candidates, candidate_query, targets = [], [], [], [], []

    for i, q in enumerate(queries):
        chosen = q["chunks"] if len(q["chunks"]) <= TRAIN_CONTEXTS else rng.sample(q["chunks"], TRAIN_CONTEXTS)
        context_chunks.extend(chosen)
        context_owner.extend([i] * len(chosen))

        targets.append(len(candidates))
        candidates.append(int(rng.choice(q["gold"])))
        candidate_query.append(i)
        for c in rng.sample(q["hard"], min(HARD_NEGATIVES, len(q["hard"]))):
            candidates.append(c)
            candidate_query.append(i)

    passage_chunks, passage_owner = [], []
    for j, (c, i) in enumerate(zip(candidates, candidate_query)):
        shown = [pool[c]["chunks"][k] for k in queries[i]["show"][c]]
        passage_chunks.extend(shown)
        passage_owner.extend([j] * len(shown))

    blocked = torch.zeros(len(queries), len(candidates), dtype=torch.bool)
    for i, q in enumerate(queries):
        relevant = set(q["all_gold"].tolist()) | set(q["banned"].tolist())
        for j, c in enumerate(candidates):
            if j != targets[i] and (c in relevant or c == candidates[targets[i]]):
                blocked[i, j] = True

    return (context_chunks + passage_chunks, context_owner, passage_owner,
            torch.tensor(targets), blocked, candidates)


def batch_logits(vectors, n_contexts, context_owner, passage_owner, n_queries, n_candidates):
    """Chunk vectors -> [n_queries, n_candidates] = best context x best passage cosine / TEMPERATURE."""

    sim = vectors[:n_contexts] @ vectors[n_contexts:].T                           # [contexts, passages]

    bounds = np.searchsorted(passage_owner, np.arange(n_candidates + 1))
    per_candidate = torch.stack([sim[:, a:b].max(dim=1).values for a, b in zip(bounds[:-1], bounds[1:])], dim=1)

    bounds = np.searchsorted(context_owner, np.arange(n_queries + 1))
    per_query = torch.stack([per_candidate[a:b].max(dim=0).values for a, b in zip(bounds[:-1], bounds[1:])])

    return per_query / TEMPERATURE


def batch_loss(vectors, batch, n_queries, teacher_vectors=None):
    chunks, context_owner, passage_owner, targets, blocked, candidates = batch
    n_contexts = len(context_owner)

    logits = batch_logits(vectors, n_contexts, context_owner, passage_owner, n_queries, len(candidates))
    logits = logits.masked_fill(blocked, -1e4)
    loss = F.cross_entropy(logits, targets)
    parts = {"contrastive": loss.item()}

    if teacher_vectors is not None:          # knowledge distillation (only the KD notebook)
        teacher_logits = batch_logits(teacher_vectors, n_contexts, context_owner, passage_owner, n_queries, len(candidates))
        teacher_logits = teacher_logits.masked_fill(blocked, -1e4)
        kl = F.kl_div(F.log_softmax(logits, dim=-1), F.log_softmax(teacher_logits, dim=-1),
                      log_target=True, reduction="batchmean")
        cosine = 1 - (vectors * teacher_vectors).sum(dim=-1).mean()
        loss = (1 - KD_ALPHA) * loss + KD_ALPHA * kl + KD_COSINE_WEIGHT * cosine
        parts.update({"kl": kl.item(), "cosine": cosine.item()})

    return loss, parts


def train_step(model, optimizer, scaler, queries, rng, teacher=None):
    """One weight update (GradCache). teacher = None or the teacher retriever (no gradients)."""

    batch = make_batch(queries, rng)
    chunks = batch[0]
    batch = (chunks, batch[1], batch[2], batch[3].to(DEVICE), batch[4].to(DEVICE), batch[5])
    groups = [list(range(k, min(k + EMBED_GROUP_CHUNKS, len(chunks)))) for k in range(0, len(chunks), EMBED_GROUP_CHUNKS)]

    # pass 1: every chunk vector without gradients (+ the teacher's vectors of the same chunks)
    with torch.no_grad():
        vectors = torch.cat([model.chunk_vectors([chunks[i] for i in g]).float() for g in groups])
        teacher_vectors = None
        if teacher is not None:
            teacher.eval()
            teacher_vectors = torch.cat([teacher.chunk_vectors([chunks[i] for i in g]).float() for g in groups])

    vectors.requires_grad_(True)
    loss, parts = batch_loss(vectors, batch, len(queries), teacher_vectors)
    loss.backward()
    grad = vectors.grad.detach()

    # pass 2: the same chunks with gradients, in groups
    for g in groups:
        part = model.chunk_vectors([chunks[i] for i in g]).float()
        scaler.scale((part * grad[g]).sum()).backward()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_([p for group in optimizer.param_groups for p in group["params"]], GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    return loss.item(), parts


def make_optimizer(model, total_steps):
    groups = [{"params": params, "lr": lr, "weight_decay": wd} for params, lr, wd in backbone_groups(model) if params]
    groups.append({"params": model.head_parameters(), "lr": LR_HEAD, "weight_decay": 0.0})

    optimizer = torch.optim.AdamW(groups)
    warmup = max(1, int(WARMUP_RATIO * total_steps))
    schedule = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: min(1.0, (s + 1) / warmup) * max(0.05, 1 - s / max(total_steps, 1)))

    return optimizer, schedule


def trainable_state(model):
    """The weights that training changes (+ the head buffers), on the CPU."""
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            if k in names or not k.startswith("backbone.")}


def check_gradcache(model):
    """The GradCache gradient must equal the gradient of one ordinary big-batch pass, and must not be 0."""

    queries = [q for q in SPLITS["train"]["queries"] if len(q["gold"]) and len(q["hard"])][:2]
    batch = make_batch(queries, random.Random(0))
    chunks = batch[0]
    batch = (chunks, batch[1], batch[2], batch[3].to(DEVICE), batch[4].to(DEVICE), batch[5])
    params = [p for p in model.parameters() if p.requires_grad]

    def grads():
        g = [p.grad.detach().clone() if p.grad is not None else torch.zeros_like(p) for p in params]
        model.zero_grad(set_to_none=True)
        return g

    model.zero_grad(set_to_none=True)
    batch_loss(model.chunk_vectors(chunks).float(), batch, len(queries))[0].backward()
    direct = grads()

    with torch.no_grad():
        v = model.chunk_vectors(chunks).float()
    v.requires_grad_(True)
    batch_loss(v, batch, len(queries))[0].backward()
    for k in range(0, len(chunks), 3):
        (model.chunk_vectors(chunks[k:k + 3]).float() * v.grad[k:k + 3]).sum().backward()
    cached = grads()

    num = sum(((a - b).float() ** 2).sum() for a, b in zip(direct, cached)).sqrt()
    den = sum((a.float() ** 2).sum() for a in direct).sqrt() + 1e-12
    print(f"GradCache check: relative gradient difference {(num / den).item():.2e} | gradient norm {den.item():.2e}")
    assert den.item() > 1e-10, "no gradient reaches the trained weights"
    assert (num / den).item() < (5e-2 if USE_AMP else 1e-3), "GradCache gradient differs from the direct gradient"


def train_retriever(model, run_name, teacher=None, epochs=None):
    """Train on TRAIN queries; dev picks the best epoch (epoch 0 = the start, no training). Returns the history."""

    epochs = EPOCHS if epochs is None else epochs
    train_queries = [q for q in SPLITS["train"]["queries"] if len(q["gold"]) and len(q["hard"])]
    steps_per_epoch = math.ceil(len(train_queries) / BATCH_QUERIES)

    optimizer, schedule = make_optimizer(model, steps_per_epoch * epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    rng = random.Random(SEED)

    result = evaluate_split(model, "dev", EPOCH_EVAL_CHUNKS)
    show("epoch 0 (no training)", result)
    history = [{"epoch": 0, "dev_f1_fused": result["fused"]["f1"], "dev_f1_model": result["model"]["f1"],
                "weight": result["weight"]}]
    # best epoch: highest dev F1@7 of the MODEL ALONE (that is what training changes; the fused score
    # hardly moves because the re-ranker carries most of it); on a tie, the higher fused F1@7
    best_key, best_state, best_epoch, bad = (round(result["model"]["f1"], 5), result["fused"]["f1"]), trainable_state(model), 0, 0

    print(f"\n{run_name} | {len(train_queries)} train queries, {steps_per_epoch} updates per epoch")
    t0 = time.time()

    for epoch in range(1, epochs + 1):

        order = train_queries[:]
        rng.shuffle(order)
        losses = []

        for step in range(steps_per_epoch):
            loss, parts = train_step(model, optimizer, scaler, order[step * BATCH_QUERIES:(step + 1) * BATCH_QUERIES], rng, teacher)
            schedule.step()
            losses.append(loss)

            if (step + 1) % 25 == 0 or step + 1 == steps_per_epoch:
                extra = " ".join(f"{k} {v:.3f}" for k, v in parts.items())
                print(f"    update {step + 1}/{steps_per_epoch} | loss {np.mean(losses[-25:]):.4f} ({extra}) | "
                      f"{(time.time() - t0) / 60:.1f} min", flush=True)

        result = evaluate_split(model, "dev", EPOCH_EVAL_CHUNKS)
        key = (round(result["model"]["f1"], 5), result["fused"]["f1"])
        improved = key > best_key
        show(f"epoch {epoch}/{epochs}" + ("  <- best" if improved else ""), result)
        history.append({"epoch": epoch, "loss": float(np.mean(losses)), "dev_f1_fused": result["fused"]["f1"],
                        "dev_f1_model": result["model"]["f1"], "weight": result["weight"]})

        if improved:
            best_key, best_state, best_epoch, bad = key, trainable_state(model), epoch, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"no improvement for {PATIENCE} epochs -> stop")
                break

    model.load_state_dict(best_state, strict=False)
    peak = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0
    print(f"\n✅ Trained in {(time.time() - t0) / 60:.1f} min | best epoch {best_epoch} | dev F1@7 model {best_key[0]:.4f}, "
          f"fused {best_key[1]:.4f} ({EPOCH_EVAL_CHUNKS} passages per candidate) | peak GPU {peak:.2f} GB")

    return {"history": history, "best_epoch": best_epoch, "train_minutes": (time.time() - t0) / 60, "peak_gpu_gb": peak}


# ------------------------------------------------------------
# Reuse the trained weights of an earlier run (retriever.pt, written by dev_report)
# ------------------------------------------------------------

def find_trained(run, path=None):
    """retriever.pt of the run `run`: the path given, or searched under /kaggle/input. None if not found."""

    if path is not None and Path(path).exists():
        return Path(path)

    root = Path("/kaggle/input")
    for p in (sorted(root.rglob("retriever.pt")) if root.exists() else []):
        try:
            if torch.load(p, map_location="cpu", weights_only=False).get("run") == run:
                return p
        except Exception:
            pass

    return None


def load_trained(model, path):
    """Load retriever.pt into a freshly built model (every weight that trains must be in the file)."""

    saved = torch.load(path, map_location="cpu", weights_only=False)
    result = model.load_state_dict(saved["state"], strict=False)
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}

    assert not result.unexpected_keys, f"unexpected weights: {result.unexpected_keys[:5]}"
    assert not trainable & set(result.missing_keys), f"trained weights missing in {path}"
    print(f"trained weights loaded from {path} (run {saved.get('run')})")

### CELL 15 · Before training: token statistics + GradCache check

**In simple words:** Measure the feature statistics used to standardize the vectors, and check that the memory-saving trick (GradCache) gives the same gradient as ordinary training.

In [15]:
# ============================================================
# CELL 15 — BEFORE TRAINING: TOKEN STATISTICS + GRADCACHE CHECK
#
# The hidden vectors of a language model have a few huge, shared dimensions;
# without standardization they dominate the cosine and every passage looks alike.
# ============================================================

def set_token_statistics(model, docs):
    """Mean / std of the mixed token vectors (pooled tokens), from passages of these train candidates."""

    model.eval()
    total, total_sq, count = 0.0, 0.0, 0
    chunks = [x for d in docs for x in select_chunks(d["chunks"], 2)]

    with torch.no_grad():
        for k in range(0, len(chunks), EVAL_CHUNKS):
            v = model.token_vectors_mean(chunks[k:k + EVAL_CHUNKS], standardize=False).double()
            total, total_sq, count = total + v.sum(0), total_sq + (v ** 2).sum(0), count + len(v)

    model.train()

    mean = total / count
    std = (total_sq / count - mean ** 2).clamp(min=1e-8).sqrt()
    model.token_mean.copy_(mean.float().to(DEVICE))
    model.token_std.copy_(std.clamp(min=1e-4 * std.max()).float().to(DEVICE))
    print(f"token statistics from {len(chunks)} train passages | the largest std is "
          f"{(std.max() / std.median()).item():.0f}x the median one")


stats_docs = random.Random(SEED).sample(SPLITS["train"]["pool"], min(STATS_DOCS, len(SPLITS["train"]["pool"])))


set_token_statistics(model, stats_docs)
check_gradcache(model)

# (the Phase-1 model without training is scored on dev as "epoch 0" at the start of CELL 16)

token statistics from 595 train passages | the largest std is 2x the median one
GradCache check: relative gradient difference 7.10e-04 | gradient norm 8.26e+00


### CELL 16 · Train (train queries only; the model's own dev F1@7 picks the best epoch)

**In simple words:** Run the training (or, with `REUSE_TRAINED = True`, load an earlier run). Epoch 0 = the Phase-1 model before training; the epoch with the best model-alone dev score is kept.

In [16]:
# ============================================================
# CELL 16 — TRAIN (TRAIN queries only; the model's own dev F1@7 picks the best epoch)
#   REUSE_TRAINED = True (CELL 2): load retriever.pt of an earlier run of this notebook instead
# ============================================================

if REUSE_TRAINED:
    trained_path = find_trained(RUN_NAME)
    assert trained_path is not None, f"no retriever.pt of run {RUN_NAME} under /kaggle/input: attach the earlier output"
    load_trained(model, trained_path)
    training = {"loaded_from": str(trained_path)}
else:
    training = train_retriever(model, RUN_NAME)      # epoch 0 = the Phase-1 model without training (zero-shot)

epoch 0 (no training)              model 0.2835 | fused 0.4725 (model weight 0.1) | re-ranker 0.4725 | BM25 0.4647 | encoded in 102s

p7_arch1_lorma | 801 train queries, 101 updates per epoch
    update 25/101 | loss 2.1039 (contrastive 1.654) | 3.4 min
    update 50/101 | loss 2.1728 (contrastive 2.091) | 6.7 min
    update 75/101 | loss 1.7912 (contrastive 1.485) | 10.0 min
    update 100/101 | loss 1.7318 (contrastive 1.736) | 13.4 min
    update 101/101 | loss 1.7827 (contrastive 2.418) | 13.4 min
epoch 1/6  <- best                 model 0.3335 | fused 0.4736 (model weight 0.2) | re-ranker 0.4725 | BM25 0.4647 | encoded in 108s
    update 25/101 | loss 1.5017 (contrastive 1.073) | 18.6 min
    update 50/101 | loss 1.6190 (contrastive 1.084) | 21.9 min
    update 75/101 | loss 1.7032 (contrastive 0.831) | 25.2 min
    update 100/101 | loss 1.4175 (contrastive 1.242) | 28.6 min
    update 101/101 | loss 1.4130 (contrastive 0.974) | 28.6 min
epoch 2/6  <- best                 model 0.

### CELL 17 · Dev report (all passages) + fusion weight for the test ranking

**In simple words:** Final dev scores with all 16 passages per candidate, and the model weight `w` for the test ranking (picked on dev).

**How it works:** The tables show BM25, the re-ranker, the model alone and the mix, recall in the top 50, and the score by number of relevant cases.

In [17]:
# ============================================================
# CELL 17 — DEV REPORT (all passages) + FUSION WEIGHT FOR THE TEST RANKING
#   (dev_report is defined in CELL 13)
# ============================================================

sizes = {"parameters_backbone": n_backbone, "parameters_trained_in_backbone": n_trained, "parameters_head": n_head}
FUSION_WEIGHT, results = dev_report(model, training, sizes)

DEV — p7_arch1_lorma (118 queries, pool 1023), 16 passages per candidate

ranking                                  micro-F1@7      P@7      R@7
BM25 alone (reference)                       0.4647   0.5061   0.4296
re-ranker alone (CELL 10)                    0.4725   0.5145   0.4368
the model alone                              0.3702   0.4031   0.3422
fused: model 0.4 + re-ranker -> test         0.4769   0.5194   0.4409
oracle (7 predictions per query)             0.6181

model weight -> dev micro-F1@7: 1.0: 0.3813, 0.9: 0.4080, 0.8: 0.4325, 0.7: 0.4491, 0.6: 0.4569, 0.5: 0.4691, 0.4: 0.4769, 0.3: 0.4747, 0.2: 0.4747, 0.1: 0.4736

relevant cases in the top 50: model 66.3% | BM25 71.8% | fused 72.5%

 relevant cases  queries  fused F1@7
            0-3       40      0.3182
            4-7       39      0.5868
           8-15       25      0.6335
            16+       14      0.3493

correct cases among the 7 predictions -> queries: {0: 3, 1: 25, 2: 22, 3: 9, 4: 12, 5: 14, 6: 16, 7: 17}


### CELL 18 · Rank the test pool + write prediction.json + submission.zip

**In simple words:** Rank every test query's pool with model + re-ranker, keep the top 7, check the format and write `submission.zip`.

**How it works:** Checks: every query present, 7 unique ids from its own pool, never the query itself.

In [18]:
# ============================================================
# CELL 18 — RANK THE TEST POOL + WRITE prediction.json + submission.zip
# ============================================================

S = SPLITS["test"]
scores, seconds = model_scores(model, "test", MAX_CHUNKS)
ranked = top_k(mask_banned(fuse(scores, S["ranker"], FUSION_WEIGHT), S["queries"]))      # model + re-ranker (CELL 10)

predictions = {q["id"]: [S["pool"][k]["id"] for k in r] for q, r in zip(S["queries"], ranked)}

# format checks (manual 13.3.5): every query, exactly 7 unique ids from ITS pool, best first
pool_ids = {c["id"] for c in S["pool"]}
assert len(predictions) == len(S["queries"]) == len({q["id"] for q in S["queries"]})
for qid, ids in predictions.items():
    assert len(ids) == TOP_K and len(set(ids)) == TOP_K, qid
    assert all(i in pool_ids for i in ids), qid
    assert canon(qid) not in {canon(i) for i in ids}, qid

PREDICTION_FILE = OUTPUT_DIR / "prediction.json"
ZIP_FILE = OUTPUT_DIR / "submission.zip"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=1)

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(PREDICTION_FILE, arcname="prediction.json")

print(f"✅ {len(predictions)} test queries x {TOP_K} candidates (model weight {FUSION_WEIGHT:.1f}) | encoded in {seconds:.0f}s")
print("submission.zip:", zipfile.ZipFile(ZIP_FILE).namelist())
print("Saved:", ZIP_FILE)

✅ 237 test queries x 7 candidates (model weight 0.4) | encoded in 319s
submission.zip: ['prediction.json']
Saved: /kaggle/working/p7_arch1_lorma/submission.zip
